# Fusion Ablation — Vision-only vs Vision + Tabular

Compares the vision-only DenseNet-121 baseline (`src/train.py`) against the
fusion model (`src/train_fusion.py`) to quantify what tabular metadata
(age, gender, view position) adds on top of the image alone.

Run both training scripts first, then this notebook.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

vision_metrics = pd.read_csv("../logs/vision_baseline/metrics.csv")
fusion_metrics = pd.read_csv("../logs/fusion/metrics.csv")

vision_metrics.tail(), fusion_metrics.tail()

## Validation Macro AUROC — training curves compared

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(vision_metrics["epoch"], vision_metrics["val_macro_auroc"], label="Vision-only", marker="o")
ax.plot(fusion_metrics["epoch"], fusion_metrics["val_macro_auroc"], label="Vision + Tabular (fusion)", marker="o")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Macro AUROC")
ax.set_title("Fusion ablation: does tabular metadata help?")
ax.legend()
plt.tight_layout()
plt.savefig("../docs/fusion_ablation_curves.png", dpi=150)
plt.show()

## Per-class AUROC comparison (test set)

In [ ]:
vision_results = pd.read_csv("../docs/vision_baseline_results.csv").set_index("class")
fusion_results = pd.read_csv("../docs/fusion_results.csv").set_index("class")

comparison = vision_results.join(fusion_results, lsuffix="_vision", rsuffix="_fusion")
comparison["improvement"] = comparison["auroc_fusion"] - comparison["auroc_vision"]
comparison.sort_values("improvement", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
comparison_sorted = comparison.sort_values("improvement")
colors = ["crimson" if v < 0 else "seagreen" for v in comparison_sorted["improvement"]]
ax.barh(comparison_sorted.index, comparison_sorted["improvement"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("AUROC improvement (fusion - vision-only)")
ax.set_title("Where does tabular fusion help or hurt, per class?")
plt.tight_layout()
plt.savefig("../docs/fusion_per_class_improvement.png", dpi=150)
plt.show()

## Notes

- Fill in the actual macro AUROC delta here once both models are trained on
  the full dataset (e.g. "fusion improved macro AUROC from X to Y").
- Worth noting in the README/report: age and view-position are plausible
  signals for certain findings (e.g. Cardiomegaly correlates with age;
  certain pathologies are easier to see on PA vs AP view), so a genuine
  improvement here isn't just noise — it's a sensible result to highlight.